# Want to verify the notes.pdf 

Need to compute the matrices and then the adjoint

In [5]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
import numpy as np
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


In [244]:
# class AdjointHook(nn.Module):
#     """
#     A wrapper module that applies a custom VJP to any given layer
#     to capture its incoming gradient (adjoint) during the backward pass.
#     """
#     layer: nn.Module  # The layer to wrap
        
#     @nn.compact
#     def __call__(self, *args, **kwargs):
#         # Define the function to which we'll attach the custom VJP
#         @jax.custom_vjp
#         def layer_with_hook(params, *args, **kwargs):
#             return self.layer.apply(params, *args, **kwargs)

#         # --- Define the Forward and Backward Passes ---
#         def layer_fwd(params, *args, **kwargs):
#             # Execute the original layer's forward pass
#             output = self.layer.apply(params, *args, **kwargs)
#             # Return output and residuals for the backward pass
#             return output, (params, args, kwargs)

#         def layer_bwd(res, g):
#             # res contains residuals: (params, args, kwargs)
#             # g is the incoming gradient (the adjoint we want to capture)
#             params, args, kwargs = res

#             # Calculate the VJP of the original wrapped layer
#             # This computes the gradients w.r.t. params and inputs
#             _, vjp_fun = jax.vjp(
#                 lambda p, *a, **kw: self.layer.apply(p, *a, **kw), params, *args, **kwargs
#             )
            
#             # The VJP function returns a tuple of gradients
#             grad_params, *grad_args = vjp_fun(g)

#             print(f"--- Captured Adjoint for layer: {self.layer.name} ---")
#             print(f'{g=}, {grad_args=}')
#             # adjoint_vals.value.append(g)
#             # adjoint_vals.value = adjoint_vals.value + [g]
#             # print(f'{adjoint_vals.value=}')
#             # adjoint_vals += g
#             print("-" * 30)
            

#             # self.adjoints.value.append(g)
            
#             # The custom VJP's backward pass must return a tuple of gradients
#             # matching the inputs of the forward pass (params, *args, **kwargs).
#             # JAX handles the kwargs gradients automatically.
#             return (grad_params,) + tuple(grad_args)

#         # Attach the custom forward and backward functions
#         layer_with_hook.defvjp(layer_fwd, layer_bwd)
        
#         # Get the parameters for the wrapped layer
#         layer_params = self.param('wrapped_layer', self.layer.init, *args, **kwargs)
#         # adjoint_vals = self.variable('mutable', 'adjoint', lambda: [])

#         return layer_with_hook(layer_params, *args, **kwargs)

class AdjointHook(nn.Module):
    """
    A wrapper module that applies a custom VJP to any given layer
    to capture its incoming gradient (adjoint) during the backward pass.
    """
    layer: nn.Module  # The layer to wrap

    @nn.compact
    def __call__(self, *args, **kwargs):
        # Define the function to which we'll attach the custom VJP
        @jax.custom_vjp
        def layer_with_hook(params, *args, **kwargs):
            jax.debug.print('here layer_with_hook')
            return self.layer.apply(params, *args, **kwargs)

        # --- Define the Forward and Backward Passes ---
        def layer_fwd(params, *args, **kwargs):
            # Execute the original layer's forward pass
            output = self.layer.apply(params, *args, **kwargs)
            jax.debug.print('here layer_fwd')
            # Return output and residuals for the backward pass
            return output, (params, args, kwargs)

        def layer_bwd(res, g):
            jax.debug.print('here backward')
            # res contains residuals: (params, args, kwargs)
            # g is the incoming gradient (the adjoint we want to capture)
            params, args, kwargs = res

            # Save the adjoint 'g' to the mutable variable
            # We need to explicitly get the variable and update its value
            self.sow('adjoints', 'captured_adjoints', '123') # Sow the adjoint

            # Calculate the VJP of the original wrapped layer
            # This computes the gradients w.r.t. params and inputs
            _, vjp_fun = jax.vjp(
                lambda p, *a, **kw: self.layer.apply(p, *a, **kw), params, *args, **kwargs
            )
            # print(dir(self))
            
            # The VJP function returns a tuple of gradients
            grad_params, *grad_args = vjp_fun(g)

            # Optional: Print for debugging
            # print(f"--- Captured Adjoint for layer: {self.layer.name} ---")
            # print(f'{g=}, {grad_args=}')
            # print("-" * 30)
            
            # The custom VJP's backward pass must return a tuple of gradients
            # matching the inputs of the forward pass (params, *args, **kwargs).
            # JAX handles the kwargs gradients automatically.
            return (grad_params,) + tuple(grad_args)

        # Attach the custom forward and backward functions
        layer_with_hook.defvjp(layer_fwd, layer_bwd)
        
        # Get the parameters for the wrapped layer
        layer_params = self.param('wrapped_layer', self.layer.init, *args, **kwargs)

        # Initialize the mutable variable to store adjoints
        # We'll use self.sow to store it in a separate 'adjoints' collection
        # and name it 'captured_adjoints' for this specific hook.
        # This creates a variable that can be accessed from the module's output.
        # self.sow('collection_name', 'variable_name', initial_value_or_callable)
        # Note: We're calling sow here, but the actual saving happens in layer_bwd
        # when 'g' is available. The initial sow just sets up the variable.
        # self.sow('adjoints', 'captured_adjoints', jnp.array([])) # Initialize with an empty array or appropriate structure
        self.sow('nested',  f'goodstufflayer_{0}_output', args[0])

        return layer_with_hook(layer_params, *args, **kwargs)

class MLP(nn.Module):
    num_units: int
    
    def setup(self):
        # self.dense1 = nn.Dense(self.num_units)
        # self.dense2 = nn.Dense(self.num_units)
        self.dense1 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
        self.dense2 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
    def __call__(self, x):
        f = self.dense1(x)
        f = nn.tanh(f)
        f = self.dense2(f)
        return f 
    

class SimpleMLP(nn.Module):
    """
    We ONLY work with "MLP" layers from above; see setup()

    Use sow to save intermediates for computation of preconditioner
    """
    num_layers: int
    num_units: int
    num_classes: int

    def setup(self):
        # Create a list of Dense layers
        self.layers = (
            *[
                AdjointHook(MLP(self.num_units, name=f"layer_{i:02})")) for i in range(self.num_layers - 1)
            ], 
            AdjointHook(MLP(self.num_units, name=f"layer_{self.num_layers-1:02}")) # For now, do all the same dimensions 
        )
        # self.layers = (
        #     *[
        #         MLP(self.num_units, name=f"layer_{i:02}") for i in range(self.num_layers - 1)
        #     ], 
        #     MLP(self.num_units, name=f"layer_{self.num_layers-1:02}") # For now, do all the same dimensions 
        # )
        
        
    def __call__(self, x):
        # Store input to match the notes
        self.sow('intermediates',  f'layer_{0}_output', x)
        
        # Pass the input through each layer
        for i, layer in enumerate(self.layers):
            x = layer(x)
            
            # Store input to match the notes
            self.sow('intermediates', f'layer_{i+1}_output', x)
            
        return x


In [245]:
# Model definition 
L = 4 
n = 1 

n_samples = 1
X = jnp.linspace(0, 1, n_samples).reshape((-1, 1)) + .1

key = jax.random.PRNGKey(0)

model = SimpleMLP(num_layers=L, num_units=n, num_classes=n)
params = model.init(key, jnp.ones((1, n)))

predictions, intermediates = model.apply(params, X, mutable=['nested'])
print(predictions, intermediates['nested'])
treescope.display(intermediates['nested'])
# Regular way
# jacobian = jax.jacobian(model.apply, argnums=0, has_aux=True)(params, X, mutable=['adjoints'])
# print(jacobian[0])
# print(jacobian[1])
# print(params)
# flattened, _ = jax.tree.flatten(jacobian)
# reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
# aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
# jacobian = aggregated_array.reshape((X.shape[0] * X.shape[1], -1))
# treescope.display(jacobian)
# treescope.display(jacobian.T @ jacobian)

here layer_with_hook
here layer_with_hook
here layer_with_hook
here layer_with_hook
here layer_with_hook
here layer_with_hook
here layer_with_hook
here layer_with_hook
[[0.00132804]] {'layers_0': {'goodstufflayer_0_output': (Array([[1.]], dtype=float64), Array([[0.1]], dtype=float64))}, 'layers_1': {'goodstufflayer_0_output': (Array([[-0.47762235]], dtype=float64), Array([[-0.06304585]], dtype=float64))}, 'layers_2': {'goodstufflayer_0_output': (Array([[0.17542378]], dtype=float64), Array([[0.02473544]], dtype=float64))}, 'layers_3': {'goodstufflayer_0_output': (Array([[0.00396423]], dtype=float64), Array([[-0.00138664]], dtype=float64))}}


In [202]:
# Create layer model; then be able to take the gradients and stuff
layer = MLP(num_units=n)

# First define function
def apply_layer(params, x): 
    # assert len(x) == 1
    # return jnp.squeeze(layer.apply(params, x))
    return layer.apply(params, x)

K_i = jax.jit(jax.jacrev(apply_layer, argnums=0))
M_i = jax.jit(jax.jacrev(apply_layer, argnums=1))

predictions, intermediates = model.apply(params, X, mutable=['intermediates'])
print(intermediates)

{'intermediates': {'layer_0_output': (Array([[0.1]], dtype=float64),), 'layer_1_output': (Array([[-0.06304585]], dtype=float64),), 'layer_2_output': (Array([[0.02473544]], dtype=float64),), 'layer_3_output': (Array([[-0.00138664]], dtype=float64),), 'layer_4_output': (Array([[0.00132804]], dtype=float64),)}}


In [30]:
 params[f'layers_{0}']['wrapped_layer']

{'params': {'dense1': {'kernel': <jax.Array([[0.99913204]], dtype=float32)>,
   'bias': <jax.Array([-0.0097766], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[-0.6213732]], dtype=float32)>,
   'bias': <jax.Array([-0.00718858], dtype=float32)>}}}

In [31]:
# %%timeit # Slightly better
# Initialize dgdu as a tridiagonal matrix with ones down the diagonal
dgdu = np.eye(n_samples * n * (L + 1)) # It's tridiagonal with ones down diagonal; will have to change once scaled up
counter = 0
for l in range(L):
    vals = -jnp.squeeze(M_i( params[f'layers_{l}']['wrapped_layer'], intermediates['intermediates'][f'layer_{l}_output'][0]))

    # vals
    for s in range(n_samples):
        dgdu[counter * (n_samples * n) + s * n:counter * (n_samples * n) + s * n + n, 
             counter * (n_samples * n) + (n_samples * n) + s * n:counter * (n_samples * n) + (n_samples * n) + s * n + n] \
                =  vals


    counter += 1
dgdu = dgdu.T
dgduinv = np.linalg.inv(dgdu)

In [33]:
dgduinv

array([[ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [-6.15817012e-01,  1.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 2.69615252e-01, -4.37817155e-01,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 9.57398003e-03, -1.55467937e-02,  3.55097865e-02,
         1.00000000e+00,  0.00000000e+00],
       [ 3.71929973e-04, -6.03961836e-04,  1.37948417e-03,
         3.88479997e-02,  1.00000000e+00]])

In [41]:
# Number of trainable parameters
p = 16
dgdt = np.zeros((p, n_samples * n * (L + 1)))
counter = 0
for l in range(L): 
    flattened, _ = jax.tree.flatten(
            K_i(params[f'layers_{l}']['wrapped_layer'], intermediates['intermediates'][f'layer_{l}_output'][0])
    )
    
    # Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
    reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
    aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
    
    layer_p = aggregated_array.shape[-1]

    for s in range(n_samples):
        dgdt[counter:counter + layer_p, 
            n_samples * n * l + n_samples * n + s * n:n_samples * n * l + n_samples * n + s * n + n] = aggregated_array[s, :].T
    counter += layer_p
    
dgdt = dgdt.T

In [67]:
dgduinv[-1, :]

array([ 3.71929973e-04, -6.03961836e-04,  1.37948417e-03,  3.88479997e-02,
        1.00000000e+00])

In [54]:
print(dgduinv)

[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [-6.15817012e-01  1.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 2.69615252e-01 -4.37817155e-01  1.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 9.57398003e-03 -1.55467937e-02  3.55097865e-02  1.00000000e+00
   0.00000000e+00]
 [ 3.71929973e-04 -6.03961836e-04  1.37948417e-03  3.88479997e-02
   1.00000000e+00]]


In [40]:
dgduinv @ dgdt

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [-6.16351962e-01, -6.16351999e-02,  1.00000000e+00,
         8.98932815e-02,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 2.69849462e-01,  2.69849479e-02, -4.37817155e-01,
        -3.93568207e-02, -2.76037753e-01,  1.74030345e-02,
         1.00000000e+00, -9.79342759e-02,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 9.58229679e-03,  9.58229737e-04, -1.55467937e-02,
        -1.39755230e-03, -9.80204166e-03,  6.17978039e-04,
         3.55097865e-02, -3.47762523e-03,  1.15589976e+00,
         2.85916850e-02,  1.00000000e+00,  7.76494053e-05,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 3.72253062e-04,  3.72253085e-05, -6.03961836e-04,
        -5.42921113e-05, -3.80789711e-04,  2.40072107e-05,
         1.37948417e-03, -1.35098784e-04,  4.49043936e-02,
         1.11072977e-03,  3.88479997e-02,  3.01652407e-06,
         3.58301520e-01, -4.96836146e-04,  1.00000000e+00,
         3.29354708e-03]])

In [39]:
jacobian

<jax.Array float32(1, 16) ≈0.09 ±0.25 [≥-0.0006, ≤1.0] nonzero:16
  <Arrayviz rendering>
| Device: GPU 0>